# Chapter 21 — Experiment Design & Power Analysis
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Design experiments with sufficient statistical power
- Compute required sample sizes for A/B tests and ML comparisons
- Understand Type I and Type II errors in practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The Four Quantities

Experiment design involves balancing four linked quantities:

| Symbol | Meaning | Typical value |
|--------|---------|---------------|
| α | Significance level (Type I error) | 0.05 |
| β | Type II error rate | 0.20 |
| 1-β | **Power** (probability of detecting true effect) | 0.80 |
| δ | Minimum Detectable Effect (MDE) | depends on domain |

Given any three, the fourth is determined. Usually we fix α, β, δ and solve for **n**.

In [ ]:
# Visualise Type I and Type II errors
delta = 0.3  # effect size
se = 1.0     # standard error
z_alpha = stats.norm.ppf(0.975)  # two-tailed α=0.05

x = np.linspace(-4, 5, 400)
null = stats.norm(0, se)
alt  = stats.norm(delta, se)

plt.figure(figsize=(10, 5))
plt.plot(x, null.pdf(x), "b-", lw=2, label="H₀: δ=0")
plt.plot(x, alt.pdf(x),  "r-", lw=2, label=f"H₁: δ={delta}")
crit_hi = z_alpha * se
plt.axvline(crit_hi, color="black", linestyle="--", label=f"Critical value={crit_hi:.2f}")
plt.fill_between(x[x >= crit_hi], null.pdf(x[x >= crit_hi]), alpha=0.4, color="blue", label="Type I (α)")
plt.fill_between(x[x < crit_hi], alt.pdf(x[x < crit_hi]), alpha=0.4, color="red", label="Type II (β)")
plt.title("Type I and Type II Errors")
plt.legend(); plt.tight_layout(); plt.show()

power = 1 - alt.cdf(crit_hi)
print(f"Power = {power:.3f}")

## 2. Math Walkthrough — Sample Size Formula

For a two-sample z-test (large n), assuming equal group sizes:
$$n = \frac{(z_{\alpha/2} + z_\beta)^2 (\sigma_1^2 + \sigma_2^2)}{\delta^2}$$

For a two-proportion test (A/B testing):
$$n = \frac{(z_{\alpha/2} + z_\beta)^2 (p_1(1-p_1) + p_2(1-p_2))}{(p_1 - p_2)^2}$$

In [ ]:
def sample_size_proportions(p1, p2, alpha=0.05, power=0.80):
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(power)
    numerator = (z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2))
    denominator = (p1 - p2)**2
    return int(np.ceil(numerator / denominator))

# Example: conversion rate from 5% to 7%
n = sample_size_proportions(0.05, 0.07)
print(f"Sample size (5% → 7%): {n} per group")

# Power curve: effect size vs sample size
effects = np.linspace(0.01, 0.05, 50)
ns = [sample_size_proportions(0.10, 0.10+e) for e in effects]
plt.plot(effects*100, ns)
plt.xlabel("Absolute Effect Size (%)"); plt.ylabel("Required n per group")
plt.title("Sample Size vs Effect Size (baseline p=10%, α=0.05, power=80%)")
plt.tight_layout(); plt.show()

In [ ]:
# Power curve: n vs power for fixed effect
n_values = np.arange(50, 2000, 25)
p1, p2 = 0.10, 0.12
z_alpha = stats.norm.ppf(0.975)

powers = []
for n in n_values:
    se_pool = np.sqrt(p1*(1-p1)/n + p2*(1-p2)/n)
    z_beta = (abs(p2-p1) / se_pool) - z_alpha
    powers.append(stats.norm.cdf(z_beta))

plt.plot(n_values, powers)
plt.axhline(0.80, color="red", linestyle="--", label="Power=0.80")
plt.axhline(0.90, color="orange", linestyle="--", label="Power=0.90")
req_n = n_values[np.searchsorted(powers, 0.80)]
plt.axvline(req_n, color="gray", linestyle=":", label=f"n={req_n}")
plt.xlabel("Sample size per group"); plt.ylabel("Power")
plt.title("Power Curve — Detect 2pp lift from 10% baseline")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Practice: full experiment design for an ML evaluation
# Suppose we want to detect 1% accuracy improvement from 85% to 86%
n_model = sample_size_proportions(0.85, 0.86)
print(f"To detect 85%→86% accuracy: {n_model} test samples per model")

# Monte Carlo power simulation
actual_power = 0
n_sim = 5000
for _ in range(n_sim):
    preds_a = rng.binomial(n_model, 0.85) / n_model
    preds_b = rng.binomial(n_model, 0.86) / n_model
    diff = preds_b - preds_a
    se = np.sqrt(preds_a*(1-preds_a)/n_model + preds_b*(1-preds_b)/n_model)
    z = diff / se if se > 0 else 0
    actual_power += (z > stats.norm.ppf(0.95))
print(f"Monte Carlo power (one-sided): {actual_power/n_sim:.3f}")